# 🎵 Spotify Listener Segmentation — K-Means Clustering

**Session 3 | Case Study 3 | DSA & ML for Business**

---

### Business Context
- Spotify has **615M+ monthly active users** across **180+ markets**
- **Discover Weekly** playlist (powered by user clustering) generated **10B+ streams**
- Users who engage with personalized playlists have **30% lower churn rate**
- Premium subscribers ($11/month) account for **87% of revenue** — segmentation drives upselling

### What You'll Learn
1. **K-Means clustering** on listening behavior features
2. **Elbow & Silhouette** analysis to select optimal K
3. **PCA visualization** for 2D cluster projection
4. **Persona naming** — Power Listener, Casual, Podcast Lover, etc.
5. **Revenue-per-segment** analysis and personalization strategy

## Step 1: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv("../datasets/spotify_listener_segmentation.csv")
print(f"Dataset: {df.shape[0]} listeners, {df.shape[1]} features")
print(f"Premium rate: {df['is_premium'].mean():.0%}")
df.describe().round(2)

## Step 2: Feature Scaling & Optimal K Selection

In [ ]:
feature_cols = ['daily_listen_hours', 'genre_diversity', 'skip_rate', 'playlist_count',
                'is_premium', 'discovery_ratio', 'sessions_per_week', 'avg_song_completion']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

# Elbow & Silhouette
K_range = range(2, 10)
inertias, sil_scores = [], []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    print(f"K={k} | Inertia: {km.inertia_:>10,.1f} | Silhouette: {silhouette_score(X_scaled, labels):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-', linewidth=2); axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia'); axes[0].grid(True, alpha=0.3)
axes[1].plot(K_range, sil_scores, 'go-', linewidth=2); axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Score'); axes[1].grid(True, alpha=0.3)
optimal_k = list(K_range)[np.argmax(sil_scores)]
axes[1].axvline(x=optimal_k, color='red', linestyle='--', label=f'Best K={optimal_k}'); axes[1].legend()
plt.tight_layout(); plt.show()

## Step 3: K-Means Clustering, Profiling & Visualization

In [ ]:
K = 5
km = KMeans(n_clusters=K, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X_scaled)

profile = df.groupby('cluster')[feature_cols].mean().round(3)

# Name personas
def name_persona(row):
    if row['daily_listen_hours'] > 3 and row['genre_diversity'] > 0.5:
        return '🎧 Power Listener'
    elif row['daily_listen_hours'] < 1.5 and row['skip_rate'] > 0.3:
        return '👂 Casual Listener'
    elif row['genre_diversity'] < 0.3 and row['avg_song_completion'] > 0.7:
        return '🏋️ Workout/Focus'
    elif row['daily_listen_hours'] > 1.5 and row['discovery_ratio'] < 0.25:
        return '🎙️ Podcast Lover'
    else:
        return '📲 Social Sharer'

profile['persona'] = profile.apply(name_persona, axis=1)
print("=== Spotify Listener Segments ===\n")
print(profile.to_string())
print(f"\nCluster sizes: {df['cluster'].value_counts().sort_index().to_dict()}")

# PCA & Visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.Set2.colors
for c in range(K):
    mask = df['cluster'] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[c]],
                    label=profile.loc[c, 'persona'], alpha=0.4, s=15)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].set_title('Spotify Listener Segments (PCA)', fontweight='bold')
axes[0].legend(fontsize=8, markerscale=3); axes[0].grid(True, alpha=0.3)

# Revenue per segment
pricing = {0: 0, 1: 10.99}  # free vs premium
df['est_monthly_rev'] = df['is_premium'].map(pricing)
rev = df.groupby('cluster')['est_monthly_rev'].agg(['mean', 'count'])
rev['persona'] = profile['persona'].values
rev.plot.bar(y='mean', ax=axes[1], color=[colors[i] for i in range(K)], legend=False)
axes[1].set_xticklabels([profile.loc[i, 'persona'] for i in range(K)], rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('Avg Monthly Revenue ($)'); axes[1].set_title('Revenue per Listener Segment', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()